# Baseline v5 — A_skip5 × 5 Epochs (Thử vượt D_skip14)

| | D_skip14 (hiện tại tốt nhất) | **A_skip5 × 5 eps** |
|--|--|--|
| SKIP_TOP_K | 14 | **5** |
| CE_EPOCHS | 5 | **5** |
| R@1 | 0.5418 | ? |
| R@5 | 0.7245 | ? (A×3eps = 0.7430) |
| MRR | 0.6307 | ? |

## Cell 0 — Config

In [ ]:
import json, csv, time, random, gc
import numpy as np, faiss, torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

ROOT, DATA_DIR = Path("."), Path(".") / "data"
EVAL_DIR = ROOT / "outputs" / "eval"
TMP_DIR  = ROOT / "outputs" / "tmp"
MDL_DIR  = ROOT / "outputs" / "models"

TRAIN_NEG    = DATA_DIR / "train_with_neg.jsonl"
DEV_FILE     = DATA_DIR / "dev.jsonl"
EVAL_QA_FILE = EVAL_DIR / "eval_qa.jsonl"
FT_BI_PATH   = MDL_DIR  / "legal_hf_finetuned" / "final"
FAISS_V4     = TMP_DIR  / "faiss_v4.index"
MAP_V4       = TMP_DIR  / "faiss_mapping_v4.jsonl"
RERANK_CSV_V4= EVAL_DIR / "rerank_metrics_v4.csv"
RESULT_CSV   = EVAL_DIR / "v5_A_skip5_5ep.csv"
CE_OUT_DIR   = MDL_DIR  / "ce_A_skip5_5ep"

BASE_CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# ── KEY CONFIG ──
SKIP_TOP_K   = 5    # ← A_skip5
CE_EPOCHS    = 5    # ← 5 epochs (fair vs D_skip14)

TOP_MINE     = 30
HARD_NEG_PER = 2
CE_BATCH     = 32
CE_MAX_LEN   = 256
TOP_N_EVAL   = 50
SEED         = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
CE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device     : {DEVICE}")
print(f"SKIP_TOP_K : {SKIP_TOP_K}  → mining range: rank {SKIP_TOP_K+1}-{TOP_MINE}")
print(f"CE_EPOCHS  : {CE_EPOCHS}")
print(f"Target to beat → R@1=0.5418, R@5=0.7245, MRR=0.6307 (D_skip14)")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path):
    rows, err = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except: err += 1
    if err: print(f"  ⚠ {err} errors")
    return rows

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False)+"\n")

def is_hit(fid, ec, mapping):
    row = mapping[fid]
    for e in ec:
        ci = e.get("chunk_index",-2)
        if ci!=-1 and row["chunk_index"]==ci: return True
        if row["van_ban"]==e.get("van_ban","") and row["dieu"]==e.get("dieu","") and row["khoan"]==e.get("khoan",""): return True
    return False

def is_pos_meta(cand, meta):
    if cand["van_ban"]==meta.get("van_ban","") and cand["van_ban"]!="" \
       and cand["dieu"]==meta.get("dieu","") and cand["khoan"]==meta.get("khoan",""): return True
    ci = meta.get("chunk_index",-2)
    return ci!=-1 and cand["chunk_index"]==ci

def avg(lst): return round(sum(lst)/len(lst),4) if lst else 0.0
print("Utilities ✓")

## Cell 2 — Load v4 Bi-Encoder + FAISS

In [ ]:
print("Loading v4 bi-encoder + FAISS...")
ft_bi      = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
print(f"  ✓ {index_v4.ntotal} vectors | {len(mapping_v4)} entries")

## Cell 3 — Mine A_skip5 negatives (rank 6-30)

In [ ]:
pos_rows = [r for r in load_jsonl(TRAIN_NEG) if r.get("label")==1]
random.seed(SEED); random.shuffle(pos_rows)
print(f"Positive rows: {len(pos_rows)}")
print(f"Mining range: rank {SKIP_TOP_K+1}-{TOP_MINE} (skip top-{SKIP_TOP_K})")

train_data = []
stats = {"pos":0, "neg":0, "no_neg":0}

for r in tqdm(pos_rows, desc=f"Mining skip={SKIP_TOP_K}"):
    query = r.get("query","").strip()
    pos_p = r.get("passage","").strip()
    meta  = r.get("meta",{})
    if not query or not pos_p: continue

    train_data.append(InputExample(texts=[query,pos_p], label=1.0))
    stats["pos"] += 1

    q_emb  = ft_bi.encode([query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_MINE)
    ids    = ids[0].tolist()

    added = 0
    for fid in ids[SKIP_TOP_K:]:      # rank SKIP_TOP_K+1 → TOP_MINE
        if fid<0 or added>=HARD_NEG_PER: break
        cand = mapping_v4[fid]
        if cand["passage"]==pos_p or is_pos_meta(cand,meta): continue
        train_data.append(InputExample(texts=[query,cand["passage"]], label=0.0))
        added += 1; stats["neg"] += 1
    if added==0: stats["no_neg"] += 1

print(f"  pos={stats['pos']}, neg={stats['neg']} ({stats['neg']/max(stats['pos'],1):.1f}/q)")
print(f"  Total: {len(train_data)} | pos_ratio: {stats['pos']/len(train_data):.0%}")

# Dev set
dev_rows = load_jsonl(DEV_FILE)
random.seed(SEED); random.shuffle(dev_rows)
dev_data = []
for r in tqdm(dev_rows[:500], desc="Dev mining", leave=False):
    query = r.get("query","").strip(); pos_p = r.get("passage","").strip()
    meta  = r.get("meta",{})
    if not query or not pos_p: continue
    dev_data.append(InputExample(texts=[query,pos_p], label=1.0))
    q_emb  = ft_bi.encode([query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_MINE)
    for fid in ids[0].tolist()[SKIP_TOP_K:]:
        if fid<0: break
        cand = mapping_v4[fid]
        if cand["passage"]==pos_p or is_pos_meta(cand,meta): continue
        dev_data.append(InputExample(texts=[query,cand["passage"]], label=0.0))
        break

print(f"Dev: {len(dev_data)} samples")

## Cell 4 — Train CE (5 epochs)
> ⏱️ ~25-35 phút (RTX 3050 Ti)

In [ ]:
# Giải phóng bi-encoder VRAM trước khi train CE
del ft_bi; gc.collect()
torch.cuda.empty_cache() if DEVICE=="cuda" else None
print("VRAM cleared ✓")

random.seed(SEED); random.shuffle(train_data)
ce = CrossEncoder(BASE_CE_MODEL, num_labels=1, max_length=CE_MAX_LEN, device=DEVICE)
evaluator = CEBinaryClassificationEvaluator.from_input_examples(dev_data, name="A_skip5")
warmup    = int(len(train_data)/CE_BATCH * CE_EPOCHS * 0.1)

print(f"Train: {len(train_data)} | Dev: {len(dev_data)} | Epochs: {CE_EPOCHS} | Warmup: {warmup}")
t0 = time.time()
ce.fit(
    train_dataloader=DataLoader(train_data, shuffle=True, batch_size=CE_BATCH),
    evaluator=evaluator, epochs=CE_EPOCHS,
    warmup_steps=warmup, output_path=str(CE_OUT_DIR),
    use_amp=(DEVICE=="cuda"),
)
elapsed = round((time.time()-t0)/60,1)
print(f"Training done in {elapsed} min")

saved = CE_OUT_DIR / "saved_model"
ce.save(str(saved))
print(f"Saved → {saved}")

## Cell 5 — Evaluate & So sánh với D_skip14

In [ ]:
gc.collect()
torch.cuda.empty_cache() if DEVICE=="cuda" else None

print("Reloading v4 bi-encoder...")
ft_bi      = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
eval_qa    = load_jsonl(EVAL_QA_FILE)
print(f"Loaded ✓ | {len(eval_qa)} questions")

r_base   = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}
r_rerank = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}

for item in tqdm(eval_qa, desc="Evaluate A_skip5×5ep"):
    query = item["query"]; ec = item["expected_citations"]
    q_emb = ft_bi.encode([query], normalize_embeddings=True,
                          convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_N_EVAL)
    ids    = ids[0].tolist()

    for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_base[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in ids[:k] if i>=0) else 0)
    mrr=0.0
    for rank,i in enumerate(ids[:10],1):
        if i>=0 and is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
    r_base["MRR@10"].append(mrr)

    cands  = [(mapping_v4[i]["passage"],i) for i in ids if i>=0]
    rscore = ce.predict([[query,c[0]] for c in cands],batch_size=32) if cands else []
    ranked = sorted(zip(rscore,[c[1] for c in cands]),reverse=True)
    r_ids  = [x[1] for x in ranked]

    for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_rerank[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in r_ids[:k]) else 0)
    mrr=0.0
    for rank,i in enumerate(r_ids[:10],1):
        if is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
    r_rerank["MRR@10"].append(mrr)

# Kết quả
D14 = {"R@1":0.5418,"R@3":0.6873,"R@5":0.7245,"MRR@10":0.6307}
A5  = {"R@1":avg(r_rerank["R@1"]),"R@3":avg(r_rerank["R@3"]),
       "R@5":avg(r_rerank["R@5"]),"MRR@10":avg(r_rerank["MRR@10"])}

print("\n" + "="*72)
print(f"  {'Metric':<10} {'D_skip14×5ep':>16} {'A_skip5×5ep':>16} {'Δ':>10}")
print("="*72)
for m in ["R@1","R@3","R@5","MRR@10"]:
    d = A5[m] - D14[m]
    win = "✅" if d > 0 else ("❌" if d < 0 else "=")
    print(f"  {m:<10} {D14[m]:>16.4f} {A5[m]:>16.4f} {d:>+9.4f} {win}")
print("="*72)

overall = sum(A5[m]-D14[m] for m in ["R@1","R@3","R@5","MRR@10"])
winner  = "A_skip5×5ep" if overall > 0 else "D_skip14×5ep"
print(f"\n★ Winner: {winner}  (Σ Δ = {overall:+.4f})")

# Lưu
rows = [{"config":"D_skip14x5ep",**D14},{"config":"A_skip5x5ep",**A5}]
with open(RESULT_CSV,"w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f,fieldnames=["config","R@1","R@3","R@5","MRR@10"])
    w.writeheader(); w.writerows(rows)
print(f"Saved → {RESULT_CSV} ✓")